<a href="https://colab.research.google.com/github/Sameekshaingole/fraud-detection-federated-learning/blob/main/Baseline_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#LOAD PREPROCESSED DATA
import numpy as np

X_train = np.load('/content/drive/MyDrive/fraud_detection_project/X_train.npy')
X_test = np.load('/content/drive/MyDrive/fraud_detection_project/X_test.npy')
y_train = np.load('/content/drive/MyDrive/fraud_detection_project/y_train.npy')
y_test = np.load('/content/drive/MyDrive/fraud_detection_project/y_test.npy')

print("Data loaded ✅")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/fraud_detection_project/X_train.npy'

In [2]:
BASELINE MODEL (CENTRALIZED)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("📊 BASELINE RESULTS")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_pred))

SyntaxError: invalid syntax (1287240824.py, line 1)

In [3]:
#SPLIT DATA INTO 3 BANKS


size = len(X_train) // 3

bank1_X, bank1_y = X_train[:size], y_train[:size]
bank2_X, bank2_y = X_train[size:2*size], y_train[size:2*size]
bank3_X, bank3_y = X_train[2*size:], y_train[2*size:]

print("Bank 1:", bank1_X.shape)
print("Bank 2:", bank2_X.shape)
print("Bank 3:", bank3_X.shape)

NameError: name 'X_train' is not defined

In [4]:
#CREATE BANK CLIENT
import flwr as fl
from sklearn.linear_model import LogisticRegression

class BankClient(fl.client.NumPyClient):
    def __init__(self, X, y):
        self.X = X
        self.y = y
        self.model = LogisticRegression(max_iter=1000)

    def get_parameters(self, config):
        return [self.model.coef_, self.model.intercept_]

    def set_parameters(self, parameters):
        self.model.coef_ = parameters[0]
        self.model.intercept_ = parameters[1]

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        self.model.fit(self.X, self.y)
        return self.get_parameters(config), len(self.X), {}

    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        loss = 1 - self.model.score(self.X, self.y)
        return loss, len(self.X), {}

ModuleNotFoundError: No module named 'flwr'

In [5]:
#INITIALIZE MODEL WEIGHTS
dummy = LogisticRegression(max_iter=1000)
dummy.fit(bank1_X, bank1_y)

initial_parameters = [dummy.coef_, dummy.intercept_]

NameError: name 'LogisticRegression' is not defined

In [6]:
#CREATE CLIENTS
client1 = BankClient(bank1_X, bank1_y)
client2 = BankClient(bank2_X, bank2_y)
client3 = BankClient(bank3_X, bank3_y)

NameError: name 'BankClient' is not defined

# New Section